# Step 5 — Settings-sweep evaluation (six configurations)

The four-method comparison (`03_skipgram.ipynb` + `04_llm_judge_evaluation.ipynb`) picked winning *methods*; this notebook tunes the two adaptive methods' *settings*. Six configurations now exist after the settings sweep — **pct_dropoff at 0.03, 0.05, 0.10** and **Kneedle at 0.3, 0.5, 1.0** — and here each is trained into a Skip-gram model and judged on its top-10 recommendations, using the **same two headline metrics** as the original comparison (Precision@10, Novel-substitute rate).

Everything about the training and the judge is reused **mechanically unchanged** from the two reference notebooks; only the *scope* is new:

- **Which configurations.** Six settings instead of four methods. The two settings that already have trained models from the original comparison (`skipgram/pct_dropoff/drop_0.10/` and `skipgram/kneedle/sensitivity_1.0/`) are **reused as-is**, not retrained; only the four new settings are trained.
- **Which anchors.** A deliberately diverse **20-anchor** set instead of the original 101. Six configurations times a large anchor sample would be expensive to judge for no real added value at this tuning stage — so the anchor set is kept small but spread across departments on purpose (see Part B).

**Outputs** (to `outputs/llm_judge_sweep/`)
- `evaluation_sample.json` — the 20-anchor set + per-configuration and intersected coverage
- `results.jsonl` — one record per judged (anchor, configuration, candidate); resume-supported
- `summary.json` — the six-row headline table
- `spotcheck.txt` — 15 sampled judgments (organic/certification oversampled)

**Requires** `gensim` (Part A) and a local Ollama server with `llama3` pulled (Part C). Author-only; run it yourself.

---
# Part A — Train Skip-gram for each of the six configurations

Identical training parameters to `03_skipgram.ipynb` (`SG=1`, `VECTOR_SIZE=128`, `WINDOW=10`, `MIN_COUNT=1`, `NEGATIVE=15`, `SAMPLE=1e-3`, `EPOCHS=20`, `WORKERS=4`, `SEED=42`), held identical across all six so any difference in results comes from the filtering value, not the training.

## A.1 Imports and paths

Paths are resolved relative to the project root so the notebook works regardless of where Jupyter is launched from.

In [1]:
import json, time, random
import datetime
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
import ollama

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data"
OUT_DIR = ROOT / "outputs"
STEP1_DIR = OUT_DIR / "step1"
FILTERS_DIR = OUT_DIR / "filters"
SKIPGRAM_DIR = OUT_DIR / "skipgram"
LLM_JUDGE_SWEEP_DIR = OUT_DIR / "llm_judge_sweep"
LLM_JUDGE_SWEEP_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir:   ", DATA_DIR)
print("Outputs dir:", OUT_DIR)

Data dir:    /Users/linnhtet/Developer/bachelor-thesis/data
Outputs dir: /Users/linnhtet/Developer/bachelor-thesis/outputs


## A.2 Parameters — the six configurations

`SG_PARAMS` are the reference notebook's training parameters verbatim. `CONFIGS` lists the six settings being compared; each records its filter-corpus folder (input), its Skip-gram folder (output), and a stable human label used in every table below. The two original settings (`drop_0.10`, `sensitivity_1.0`) are detected as already-trained and reused; the other four are trained fresh.

In [2]:
# ---- Word2Vec / SGNS parameters — IDENTICAL to 03_skipgram.ipynb, across all six ----
SG = 1                # 1 = Skip-gram (not CBOW), per Item2Vec
VECTOR_SIZE = 128     # embedding dimensionality
WINDOW = 10           # >= max sentence length -> every in-sentence pair is a positive
MIN_COUNT = 1         # CRITICAL: keep every product; gensim default 5 drops sparse anchors
NEGATIVE = 15         # negative samples per positive
SAMPLE = 1e-3         # frequent-token subsampling
EPOCHS = 20
WORKERS = 4
SEED = 42

# The six configurations: (method, value) with its filter folder and skipgram folder.
def _cfg(method, value, folder, vfmt):
    return {"method": method, "value": value, "folder": folder,
            "label": f"{method} ({vfmt})"}

CONFIGS = [
    _cfg("pct_dropoff", 0.03, "pct_dropoff/drop_0.03",      "0.03"),
    _cfg("pct_dropoff", 0.05, "pct_dropoff/drop_0.05",      "0.05"),
    _cfg("pct_dropoff", 0.10, "pct_dropoff/drop_0.10",      "0.10"),
    _cfg("kneedle",     0.3,  "kneedle/sensitivity_0.3",    "0.3"),
    _cfg("kneedle",     0.5,  "kneedle/sensitivity_0.5",    "0.5"),
    _cfg("kneedle",     1.0,  "kneedle/sensitivity_1.0",    "1.0"),
]

# A configuration is reused (not retrained) iff its skipgram model already exists.
for c in CONFIGS:
    sg_dir = SKIPGRAM_DIR / c["folder"]
    c["reuse"] = (sg_dir / "model.gensim").exists() and (sg_dir / "top10.jsonl").exists()

# The two continuity anchors, always in the evaluation set (Part B): one common
# well-connected product, one thin/orphan one. Same product_ids as Steps 1-4.
CONTINUITY_IDS = {4210: "Whole Milk", 16268: "Vegan Butter"}

print(f"{'label':<22}{'filter corpus':<30}{'status'}")
print("-" * 66)
for c in CONFIGS:
    print(f"{c['label']:<22}{c['folder']:<30}{'reuse existing' if c['reuse'] else 'train fresh'}")
print(f"\nto train fresh: {[c['label'] for c in CONFIGS if not c['reuse']] or '(none)'}")

label                 filter corpus                 status
------------------------------------------------------------------
pct_dropoff (0.03)    pct_dropoff/drop_0.03         train fresh
pct_dropoff (0.05)    pct_dropoff/drop_0.05         train fresh
pct_dropoff (0.10)    pct_dropoff/drop_0.10         reuse existing
kneedle (0.3)         kneedle/sensitivity_0.3       train fresh
kneedle (0.5)         kneedle/sensitivity_0.5       train fresh
kneedle (1.0)         kneedle/sensitivity_1.0       reuse existing

to train fresh: ['pct_dropoff (0.03)', 'pct_dropoff (0.05)', 'kneedle (0.3)', 'kneedle (0.5)']


## A.3 Train (or reuse) one model per configuration

For each configuration that needs training, the corpus is built exactly as `03_skipgram.ipynb` does (one sentence per walk record, duplicates and order kept as-is), trained with the identical parameters, and its five artifacts written to `skipgram/{method}/{value_folder}/` (`model.gensim`, `embeddings.npy`, `product_id_to_index.json`, `config.json`, `top10.jsonl`). The two already-trained configurations are left untouched; we only read their `product_id_to_index.json` back for the coverage check in Part B (their `top10.jsonl` is already on disk for Part C). `top10.jsonl` is the reference's export verbatim: every vocabulary product's top-10 by cosine similarity, excluding itself.

In [3]:
def build_corpus(folder):
    """Load a configuration's kept sentences: one sentence per walk record,
    duplicates and order kept exactly as-is (same as 03_skipgram.ipynb)."""
    records = []
    with open(FILTERS_DIR / folder / "walk_sentences.jsonl") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return [[str(pid) for pid in rec["sentence"]] for rec in records]


def p3_top_k(p3_emb, p3_idx_to_pid, sg_pid_to_idx, pid, k=10):
    """Top-k Step 3 neighbours of pid (excluding itself), as pids. Verbatim logic
    from 03_skipgram.ipynb's export."""
    sims = p3_emb @ p3_emb[sg_pid_to_idx[pid]]
    out = []
    for i in np.argsort(-sims):
        npid = p3_idx_to_pid[int(i)]
        if npid == pid:
            continue
        out.append(npid)
        if len(out) == k:
            break
    return out


config_pidx = {}   # label -> {pid: idx}  (used for coverage in Part B)

for c in CONFIGS:
    label, folder = c["label"], c["folder"]
    sg_dir = SKIPGRAM_DIR / folder
    print("=" * 78)
    print(f"CONFIG: {label}    (folder: {folder})")
    print("=" * 78)

    if c["reuse"]:
        with open(sg_dir / "product_id_to_index.json") as f:
            config_pidx[label] = {int(pid): i for pid, i in json.load(f).items()}
        print(f"reused existing model — {len(config_pidx[label]):,} products with a vector "
              f"(no retrain; top10.jsonl already on disk)")
        continue

    corpus = build_corpus(folder)

    # ---- Corpus diagnostics (same as the reference) ----
    lengths = [len(s) for s in corpus]
    token_counts = Counter(tok for s in corpus for tok in s)
    total_tokens = sum(lengths)
    print(f"training sentences     : {len(corpus):,}")
    print(f"sentence length min/mean/median/max: "
          f"{min(lengths)} / {np.mean(lengths):.2f} / {int(np.median(lengths))} / {max(lengths)}")
    print(f"distinct product tokens: {len(token_counts):,}")
    print(f"total tokens           : {total_tokens:,}")

    # ---- Train (identical parameters for every configuration) ----
    model = Word2Vec(
        sentences=corpus,
        sg=SG,
        vector_size=VECTOR_SIZE,
        window=WINDOW,
        min_count=MIN_COUNT,
        negative=NEGATIVE,
        sample=SAMPLE,
        epochs=EPOCHS,
        workers=WORKERS,
        seed=SEED,
    )
    print(f"vocabulary (products with a vector): {len(model.wv):,}")
    print(f"embedding matrix shape             : {model.wv.vectors.shape}")

    # ---- Save the five artifacts ----
    sg_dir.mkdir(parents=True, exist_ok=True)
    tokens = list(model.wv.index_to_key)
    sg_embeddings = model.wv.vectors                    # (vocab, VECTOR_SIZE), raw (unnormalised)
    sg_pid_to_idx = {int(tok): i for i, tok in enumerate(tokens)}

    model.save(str(sg_dir / "model.gensim"))
    np.save(sg_dir / "embeddings.npy", sg_embeddings)
    with open(sg_dir / "product_id_to_index.json", "w") as f:
        json.dump({str(pid): i for pid, i in sg_pid_to_idx.items()}, f)

    config = {
        "method": c["method"],
        "value": c["value"],
        "sg": SG, "vector_size": VECTOR_SIZE, "window": WINDOW, "min_count": MIN_COUNT,
        "negative": NEGATIVE, "sample": SAMPLE, "epochs": EPOCHS, "workers": WORKERS, "seed": SEED,
        "n_sentences": len(corpus),
        "n_distinct_tokens": len(token_counts),
        "total_tokens": total_tokens,
        "vocab_size": len(model.wv),
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    with open(sg_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    # ---- Export top-10 for every vocabulary token (verbatim from the reference) ----
    p3_emb = sg_embeddings / np.clip(np.linalg.norm(sg_embeddings, axis=1, keepdims=True), 1e-12, None)
    p3_idx_to_pid = {i: int(tok) for i, tok in enumerate(tokens)}
    with open(sg_dir / "top10.jsonl", "w") as f:
        for pid in [int(tok) for tok in tokens]:
            neighbours = p3_top_k(p3_emb, p3_idx_to_pid, sg_pid_to_idx, pid, 10)
            f.write(json.dumps({"anchor_id": pid, "neighbours": neighbours}) + "\n")

    config_pidx[label] = sg_pid_to_idx
    print(f"Saved skipgram/{folder}/ (model.gensim, embeddings.npy, "
          f"product_id_to_index.json, config.json, top10.jsonl)")

print("\nAll six configurations ready (trained or reused).")

CONFIG: pct_dropoff (0.03)    (folder: pct_dropoff/drop_0.03)
training sentences     : 38,260
sentence length min/mean/median/max: 2 / 2.24 / 2 / 10
distinct product tokens: 17,723
total tokens           : 85,741
vocabulary (products with a vector): 17,723
embedding matrix shape             : (17723, 128)
Saved skipgram/pct_dropoff/drop_0.03/ (model.gensim, embeddings.npy, product_id_to_index.json, config.json, top10.jsonl)
CONFIG: pct_dropoff (0.05)    (folder: pct_dropoff/drop_0.05)
training sentences     : 59,152
sentence length min/mean/median/max: 2 / 2.27 / 2 / 13
distinct product tokens: 23,007
total tokens           : 134,188


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary (products with a vector): 23,007
embedding matrix shape             : (23007, 128)
Saved skipgram/pct_dropoff/drop_0.05/ (model.gensim, embeddings.npy, product_id_to_index.json, config.json, top10.jsonl)
CONFIG: pct_dropoff (0.10)    (folder: pct_dropoff/drop_0.10)
reused existing model — 26,814 products with a vector (no retrain; top10.jsonl already on disk)
CONFIG: kneedle (0.3)    (folder: kneedle/sensitivity_0.3)
training sentences     : 81,962
sentence length min/mean/median/max: 2 / 2.33 / 2 / 16
distinct product tokens: 28,048
total tokens           : 190,685


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


vocabulary (products with a vector): 28,048
embedding matrix shape             : (28048, 128)
Saved skipgram/kneedle/sensitivity_0.3/ (model.gensim, embeddings.npy, product_id_to_index.json, config.json, top10.jsonl)
CONFIG: kneedle (0.5)    (folder: kneedle/sensitivity_0.5)
training sentences     : 88,414
sentence length min/mean/median/max: 2 / 2.35 / 2 / 16
distinct product tokens: 28,867
total tokens           : 208,109
vocabulary (products with a vector): 28,867
embedding matrix shape             : (28867, 128)
Saved skipgram/kneedle/sensitivity_0.5/ (model.gensim, embeddings.npy, product_id_to_index.json, config.json, top10.jsonl)
CONFIG: kneedle (1.0)    (folder: kneedle/sensitivity_1.0)
reused existing model — 35,458 products with a vector (no retrain; top10.jsonl already on disk)

All six configurations ready (trained or reused).


---
# Part B — A deliberately diverse 20-anchor evaluation set

Not a random sample. The original six stress-test anchors were **all dairy**, and that blind spot is exactly what let oil/vinegar and household-item leakage go unnoticed under Kneedle until a manual read of example walks caught it. This evaluation set is designed not to repeat that mistake: it spreads across departments on purpose.

- **Always included:** `4210` (Whole Milk) and `16268` (Vegan Butter) — one common well-connected product, one thin/orphan one — for continuity with everything validated so far.
- **The remaining 18:** each candidate starting product's department is read **directly from `products_text.csv`'s `department` column** (not assumed from memory), and the draw is **stratified round-robin** — one product per distinct department per pass, `np.random.default_rng(42)` — maximising distinct-department coverage rather than sampling randomly across the whole pool.
- **Safety check:** after the draw we explicitly verify the set covers at least one **pantry/condiments** product (oil, vinegar, condiments) and at least one **household or personal-care** product — the exact categories known to have caused problems — and deliberately substitute one in if the round-robin didn't already include it.

## B.1 Build the stratified 20-anchor set

In [4]:
# ---- Product -> department, straight from products_text.csv (not assumed) ----
products_text = pd.read_csv(STEP1_DIR / "products_text.csv")
id_to_name = dict(zip(products_text["product_id"], products_text["product_name"]))
pid_to_dept = {int(p): d for p, d in zip(products_text["product_id"], products_text["department"])}


def name(pid):
    return id_to_name.get(int(pid), f"?{pid}")


# The departments known to have caused leakage before — the safety categories.
PANTRY_DEPTS = {"pantry"}                    # oil, vinegar, condiments, spices live here
HOUSEHOLD_DEPTS = {"household", "personal care"}

# ---- Candidate pool: the walk's own starting products (the intended anchors) ----
with open(OUT_DIR / "shared_sample.json") as f:
    shared_sample = json.load(f)
starting_products = [int(p) for p in shared_sample["starting_products"]]

FIXED = [4210, 16268]   # the two continuity anchors, always included
rng = np.random.default_rng(42)

# Group the (non-fixed) starting products by department; shuffle each group and the
# department visiting order deterministically with the seeded rng.
from collections import defaultdict
dept_to_pids = defaultdict(list)
for pid in starting_products:
    if pid in FIXED:
        continue
    d = pid_to_dept.get(pid)
    if d is None or (isinstance(d, float) and np.isnan(d)):
        continue
    dept_to_pids[d].append(pid)

dept_order = sorted(dept_to_pids)           # deterministic base order ...
rng.shuffle(dept_order)                     # ... then a seeded shuffle
for d in dept_order:
    rng.shuffle(dept_to_pids[d])

# ---- Round-robin: one product per distinct department per pass, until 18 ----
N_STRAT = 18
selected = []
progressed = True
while len(selected) < N_STRAT and progressed:
    progressed = False
    for d in dept_order:
        if len(selected) >= N_STRAT:
            break
        if dept_to_pids[d]:
            selected.append(dept_to_pids[d].pop())
            progressed = True

print(f"Round-robin drew {len(selected)} products across "
      f"{len({pid_to_dept.get(p) for p in selected})} distinct departments.")

# ---- Safety substitution: guarantee pantry/condiments + household/personal-care ----
def pick_in(dept_set, exclude):
    """A product in one of dept_set, preferring the walk's starting products, else
    anywhere in the catalogue; deterministic. None if the catalogue has none."""
    starts = [p for p in starting_products if p not in exclude and pid_to_dept.get(p) in dept_set]
    if starts:
        return int(rng.choice(np.array(starts)))
    catalogue = [int(p) for p in products_text["product_id"]
                 if int(p) not in exclude and pid_to_dept.get(int(p)) in dept_set]
    return int(rng.choice(np.array(catalogue))) if catalogue else None


protected = set(FIXED)   # never evict these when substituting
for dept_set, lbl in [(PANTRY_DEPTS, "pantry/condiments"),
                      (HOUSEHOLD_DEPTS, "household/personal-care")]:
    present = any(pid_to_dept.get(p) in dept_set for p in selected + FIXED)
    if present:
        print(f"  {lbl:<26} already covered by the round-robin draw.")
        continue
    exclude = set(selected) | protected
    cand = pick_in(dept_set, exclude)
    if cand is None:
        print(f"  {lbl:<26} WARNING: no product found in the catalogue, cannot add.")
        continue
    victims = [p for p in selected if p not in protected]
    victim = victims[-1]
    selected.remove(victim)
    selected.append(cand)
    protected.add(cand)
    print(f"  {lbl:<26} NOT in the draw -> substituted in {name(cand)} "
          f"({pid_to_dept.get(cand)}), evicting {name(victim)} ({pid_to_dept.get(victim)}).")

# ---- Final 20-anchor set: the two fixed continuity anchors + the stratified 18 ----
eval_anchors = FIXED + selected
assert len(eval_anchors) == 20, f"expected 20 anchors, got {len(eval_anchors)}"

print(f"\nFinal evaluation set — {len(eval_anchors)} anchors "
      f"across {len({pid_to_dept.get(p) for p in eval_anchors})} departments:")
print("=" * 70)
for p in eval_anchors:
    tag = "  (continuity)" if p in FIXED else ""
    print(f"  {p:>6}  {pid_to_dept.get(p):<16} {name(p)}{tag}")

Round-robin drew 18 products across 18 distinct departments.
  pantry/condiments          already covered by the round-robin draw.
  household/personal-care    already covered by the round-robin draw.

Final evaluation set — 20 anchors across 18 departments:
    4210  dairy eggs       Whole Milk  (continuity)
   16268  dairy eggs       Vegan Butter  (continuity)
   16434  pantry           Dark Chocolate Brownie Mix
   43822  frozen           Veggie Sausage, Egg &Cheese Breakfast Sandwich
   29109  other            H2O Closer Personal Lubricant
   10349  snacks           Gluten Free Maple Pecan Chocolate Chip Cookies
   30888  international    Jalapeno Slices Pickled
   37880  missing          Gelous Grip Hairbrush Frizz-Free + Anti-Static Smoothing
   29101  dairy eggs       Balanced Breaks Pepper Jack Natural Cheese Honey Roasted Peanuts & Raisins Snacks
    6332  canned goods     Organic Sliced Portobello Mushrooms
    7230  dry goods pasta  Ramen Noodle Soup, Beef Flavor, Family Pac

## B.2 Vector coverage across all six configurations

A configuration's vocabulary only contains products that appeared in at least one of its own kept sentences, so an anchor can have a vector under some settings but not others (this is exactly what happened to Whole Milk under the base disparity run in the original comparison). We report per-configuration coverage of the 20 anchors and the **intersection** — the anchors that have a vector under *all six* settings, the fair apples-to-apples set, built the same way `evaluation_sample.json`'s `all_four_coverage` was. If the intersection is meaningfully smaller than 20, that is surfaced explicitly rather than dropping anchors silently.

In [5]:
print("Per-configuration coverage of the 20-anchor set:")
print("=" * 60)
coverage_by_config = {}
for c in CONFIGS:
    pidx = config_pidx[c["label"]]
    covered = [p for p in eval_anchors if p in pidx]
    coverage_by_config[c["label"]] = covered
    print(f"  {c['label']:<22} {len(covered):>3}/20  "
          f"({100 * len(covered) / 20:.0f}%)")

# Intersection: anchors with a vector under ALL six configurations.
all_six_coverage = [p for p in eval_anchors
                    if all(p in config_pidx[c["label"]] for c in CONFIGS)]
dropped = [p for p in eval_anchors if p not in all_six_coverage]

print(f"\n  all-six coverage (fair comparison set): {len(all_six_coverage)}/20")
if dropped:
    print(f"  {len(dropped)} anchor(s) NOT covered by every configuration "
          f"(excluded from the fair set, surfaced here rather than dropped silently):")
    for p in dropped:
        missing = [c["label"] for c in CONFIGS if p not in config_pidx[c["label"]]]
        print(f"    {p:>6}  {name(p):<40} missing under: {', '.join(missing)}")
    if len(dropped) >= 5:
        print("  NOTE: the intersection is meaningfully smaller than 20 — coverage, "
              "not just score, is a real difference between these settings.")
else:
    print("  every one of the 20 anchors is covered by all six configurations.")

evaluation_sample = {
    "eval_anchors": eval_anchors,
    "continuity_anchors": FIXED,
    "coverage_by_config": {label: covered for label, covered in coverage_by_config.items()},
    "all_six_coverage": all_six_coverage,
    "dropped_from_fair_set": dropped,
    "n_departments": len({pid_to_dept.get(p) for p in eval_anchors}),
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(LLM_JUDGE_SWEEP_DIR / "evaluation_sample.json", "w") as f:
    json.dump(evaluation_sample, f, indent=2)
print(f"\nSaved {LLM_JUDGE_SWEEP_DIR.name}/evaluation_sample.json")

Per-configuration coverage of the 20-anchor set:
  pct_dropoff (0.03)      12/20  (60%)
  pct_dropoff (0.05)      13/20  (65%)
  pct_dropoff (0.10)      19/20  (95%)
  kneedle (0.3)           17/20  (85%)
  kneedle (0.5)           17/20  (85%)
  kneedle (1.0)           19/20  (95%)

  all-six coverage (fair comparison set): 11/20
  9 anchor(s) NOT covered by every configuration (excluded from the fair set, surfaced here rather than dropped silently):
     29109  H2O Closer Personal Lubricant            missing under: pct_dropoff (0.03), pct_dropoff (0.05), kneedle (0.3), kneedle (0.5), kneedle (1.0)
     30888  Jalapeno Slices Pickled                  missing under: pct_dropoff (0.03), pct_dropoff (0.05), kneedle (0.3), kneedle (0.5)
     37880  Gelous Grip Hairbrush Frizz-Free + Anti-Static Smoothing missing under: pct_dropoff (0.03), pct_dropoff (0.05), pct_dropoff (0.10)
      6332  Organic Sliced Portobello Mushrooms      missing under: pct_dropoff (0.03), pct_dropoff (0.05)
     4

---
# Part C — Judge and score

Same judge mechanics as `04_llm_judge_evaluation.ipynb`: the corrected `SYSTEM_PROMPT` (with the organic carve-out fix already applied), `build_user_prompt`, `resolve_verdict`, and `judge_pair` with retry logic — all reused verbatim. The two headline metrics are computed with the identical macro-averaged definitions (per anchor first, then averaged): **Precision@10** and **Novel-substitute rate**.

## C.1 The corrected judge prompt and judging mechanics

Reused **verbatim** from `04_llm_judge_evaluation.ipynb`, including the organic / certification carve-out fix merged into the first paragraph. Do not revert.

In [7]:
# Judge configuration — defined BEFORE the reused judge mechanics below, which
# reference MAX_RETRIES as a default argument (evaluated at def time). Same
# values as 04_llm_judge_evaluation.ipynb.
MODEL_NAME = "llama3"
MAX_RETRIES = 2

In [8]:
SYSTEM_PROMPT = """You are a grocery shopping assistant helping evaluate whether one \
product could reasonably substitute another when the original is unavailable.

You will be given two product names: the ORIGINAL product a customer wants, and a \
CANDIDATE product that might be offered instead if the original is out of stock.

First, decide whether this pair differs specifically by ONE product being a reduced, \
restricted, or "free-from" version of an attribute the OTHER has in its standard \
form. Examples: fat content (fat-free vs whole), caffeine (decaf vs regular), sugar \
(sugar-free vs regular), sweetness (unsweetened vs sweetened), alcohol \
(non-alcoholic vs regular), gluten (gluten-free vs regular). This does NOT include \
"organic", "natural", "cage-free", or similar certification or preference labels, \
even though they can look similar at a glance — treat these the same as a flavor \
or brand difference: usually an acceptable substitute in either direction, since \
most shoppers treat them as a preference rather than a requirement. Example: \
"Organic Whole Milk" vs "Whole Milk" is NOT a restricted-attribute pair \
(is_directional_pair: false), the same as two different brands of the same \
product would be.

If this pair IS this kind of restricted-attribute pair, set "is_directional_pair" \
to true, and set "restricted_item" to either "original" or "candidate", whichever \
one is the reduced/restricted/free-from version. Your "is_substitute" field will be \
ignored for this kind of pair, so just make your best guess.

If this pair is NOT this kind of restricted-attribute pair (a flavor difference, \
brand difference, or genuinely different product), set "is_directional_pair" to \
false, "restricted_item" to null, and judge "is_substitute" directly using this \
guidance:
- Different flavors, scents, or varieties of the exact same type of item (two \
flavors of ice cream, juice, soda, chips, etc.) are ALWAYS acceptable substitutes \
for each other. Only say false here if the candidate is a genuinely different \
product, not just a different flavor.
- Different brands of the same type of product are usually acceptable substitutes.
- Say false mainly when the candidate is a genuinely different product category or \
use case.

Respond with a JSON object only, in exactly this format, with no other text:
{"reason": "one short phrase, under 10 words", "is_directional_pair": true or false, \
"restricted_item": "original" or "candidate" or null, "is_substitute": true or false}"""

def build_user_prompt(anchor, candidate):
    return (
        f'Original product: "{anchor}"\n'
        f'Candidate product: "{candidate}"\n\n'
        f'If a customer wanted to buy "{anchor}" but it was out of stock, '
        f'would "{candidate}" be an acceptable substitute?'
    )

def resolve_verdict(parsed):
    """For restricted-attribute pairs, apply the rule ourselves in code, so it's
    never at the mercy of the model getting the branch backwards."""
    if parsed.get("is_directional_pair") and parsed.get("restricted_item") in ("original", "candidate"):
        # Acceptable only if the CANDIDATE is the restricted one (offering someone
        # who wanted standard a slightly-reduced version).
        # Not acceptable if the ORIGINAL is restricted (offering someone who
        # specifically wanted restricted the plain standard version instead).
        return parsed["restricted_item"] == "candidate"
    return parsed["is_substitute"]

# ---- Ollama call with defensive parsing ----
def judge_pair(anchor, candidate, retries=MAX_RETRIES):
    for attempt in range(retries + 1):
        try:
            response = ollama.chat(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(anchor, candidate)},
                ],
                format="json",
                options={"temperature": 0},
            )
            content = response["message"]["content"].strip()
            content = content.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            parsed = json.loads(content)

            if not isinstance(parsed.get("is_substitute"), bool):
                raise ValueError(f"is_substitute not a bool: {parsed}")

            final_verdict = resolve_verdict(parsed)

            return {
                "is_substitute": final_verdict,
                "raw_is_substitute": parsed["is_substitute"],
                "is_directional_pair": parsed.get("is_directional_pair", False),
                "restricted_item": parsed.get("restricted_item"),
                "reason": str(parsed.get("reason", ""))[:200],
                "parse_error": False,
            }
        except Exception as e:
            if attempt == retries:
                return {"is_substitute": None, "raw_is_substitute": None,
                         "is_directional_pair": None, "restricted_item": None,
                         "reason": f"PARSE_FAILED: {e}", "parse_error": True}
            time.sleep(0.5)

## C.2 Config, load the six top-10s, build the judged pairs, estimate time

The anchors judged are the **all-six intersection** (the fair set), plus the two continuity anchors forced in the same way the original notebook always kept its stress anchors — so every configuration is scored on the identical anchor set, with Whole Milk and Vegan Butter present wherever they have a vector. Then each configuration's `top10.jsonl` is loaded and unrolled into one long-format table: one row per (anchor, configuration, ranked candidate). A configuration contributes however many candidates an anchor has (<= 10). This is a much smaller run than the original (up to 20 x 6 x 10 = 1,200 pairs vs 4,040), but still worth a quick timing check rather than assuming.

In [9]:
RANDOM_SEED = 42

RESULTS_PATH = LLM_JUDGE_SWEEP_DIR / "results.jsonl"
SUMMARY_PATH = LLM_JUDGE_SWEEP_DIR / "summary.json"
SPOTCHECK_PATH = LLM_JUDGE_SWEEP_DIR / "spotcheck.txt"

# ---- Anchors judged: the fair all-six intersection + the two continuity anchors ----
judged_anchors = set(all_six_coverage) | set(FIXED)
print(f"anchors to judge : {len(judged_anchors)} "
      f"(all-six intersection {len(all_six_coverage)} + continuity anchors)")

# ---- Build the long-format table across all six configurations ----
long_rows = []
for c in CONFIGS:
    path = SKIPGRAM_DIR / c["folder"] / "top10.jsonl"
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            aid = int(rec["anchor_id"])
            if aid not in judged_anchors:
                continue
            for rank, cid in enumerate(rec["neighbours"], start=1):   # however many exist (<=10)
                cid = int(cid)
                long_rows.append({
                    "anchor_id": aid,
                    "anchor_name": name(aid),
                    "method": c["method"],
                    "value": c["value"],
                    "label": c["label"],
                    "rank": rank,
                    "candidate_id": cid,
                    "candidate_name": name(cid),
                })

long_df = pd.DataFrame(long_rows)
n_pairs = len(long_df)
print(f"total pairs      : {n_pairs:,} (anchors x configs x candidates-per-anchor; "
      f"upper bound 20 x 6 x 10 = 1,200)")
print("pairs per configuration:")
print(long_df.groupby("label").size().to_string())
lo, hi = n_pairs * 1 / 60, n_pairs * 2 / 60
print(f"\nrough time @1-2s/call: ~{lo:.0f}-{hi:.0f} min "
      f"(informational only — the loop below has resume support and can be stopped/continued)")

anchors to judge : 11 (all-six intersection 11 + continuity anchors)
total pairs      : 660 (anchors x configs x candidates-per-anchor; upper bound 20 x 6 x 10 = 1,200)
pairs per configuration:
label
kneedle (0.3)         110
kneedle (0.5)         110
kneedle (1.0)         110
pct_dropoff (0.03)    110
pct_dropoff (0.05)    110
pct_dropoff (0.10)    110

rough time @1-2s/call: ~11-22 min (informational only — the loop below has resume support and can be stopped/continued)


## C.3 The judging loop (resume-supported)

Same append-and-flush resume pattern as the reference. The resume key now includes `value` as well as `method` — because `pct_dropoff` appears three times across the sweep, `(anchor_id, method, candidate_id)` alone would no longer be unique.

In [10]:
# ---- Resume: skip (anchor_id, method, value, candidate_id) quads already judged ----
already_done = set()
if RESULTS_PATH.exists():
    with open(RESULTS_PATH) as f:
        for line in f:
            r = json.loads(line)
            already_done.add((r["anchor_id"], r["method"], r["value"], r["candidate_id"]))
    print(f"Resuming: {len(already_done):,} pairs already judged, will skip those.\n")

# ---- Main loop ----
with open(RESULTS_PATH, "a") as f_out:
    for i, row in long_df.iterrows():
        key = (int(row["anchor_id"]), row["method"], row["value"], int(row["candidate_id"]))
        if key in already_done:
            continue

        result = judge_pair(row["anchor_name"], row["candidate_name"])
        record = {
            "anchor_id": int(row["anchor_id"]),
            "anchor_name": row["anchor_name"],
            "method": row["method"],
            "value": row["value"],
            "label": row["label"],
            "rank": int(row["rank"]),
            "candidate_id": int(row["candidate_id"]),
            "candidate_name": row["candidate_name"],
            **result,
        }
        f_out.write(json.dumps(record) + "\n")
        f_out.flush()

        if i % 25 == 0:
            print(f"  {i:,} / {len(long_df):,} pairs processed")

print("\nDone judging.")

  0 / 660 pairs processed
  25 / 660 pairs processed
  50 / 660 pairs processed
  75 / 660 pairs processed
  100 / 660 pairs processed
  125 / 660 pairs processed
  150 / 660 pairs processed
  175 / 660 pairs processed
  200 / 660 pairs processed
  225 / 660 pairs processed
  250 / 660 pairs processed
  275 / 660 pairs processed
  300 / 660 pairs processed
  325 / 660 pairs processed
  350 / 660 pairs processed
  375 / 660 pairs processed
  400 / 660 pairs processed
  425 / 660 pairs processed
  450 / 660 pairs processed
  475 / 660 pairs processed
  500 / 660 pairs processed
  525 / 660 pairs processed
  550 / 660 pairs processed
  575 / 660 pairs processed
  600 / 660 pairs processed
  625 / 660 pairs processed
  650 / 660 pairs processed

Done judging.


## C.4 Aggregate into the two headline metrics, per configuration

Both metrics are **macro-averaged** (per anchor first, then averaged across anchors), same definitions as `04_llm_judge_evaluation.ipynb`. **Precision@10** is confirmed substitutes / candidates judged for that anchor. **Novel-substitute rate** is, among an anchor's confirmed substitutes, the fraction sitting beyond Step 1's own top-10 (rank > 10), averaged over anchors that had at least one confirmed substitute. The summary table has one row per configuration.

In [11]:
# ---- Step 1 embeddings for the novel-substitute (rank > 10) check (reused verbatim) ----
p1_emb = np.load(STEP1_DIR / "embeddings.npy").astype(np.float32)
p1_emb = p1_emb / np.clip(np.linalg.norm(p1_emb, axis=1, keepdims=True), 1e-12, None)
with open(STEP1_DIR / "product_id_to_index.json") as f:
    p1_pid_to_idx = {int(pid): row for pid, row in json.load(f).items()}

_rank_cache = {}


def p1_rank_vec(anchor_id):
    anchor_id = int(anchor_id)
    if anchor_id not in _rank_cache:
        if anchor_id not in p1_pid_to_idx:
            _rank_cache[anchor_id] = None
        else:
            sims = p1_emb @ p1_emb[p1_pid_to_idx[anchor_id]]
            ranks = np.empty(len(sims), dtype=int)
            ranks[np.argsort(-sims)] = np.arange(len(sims))
            _rank_cache[anchor_id] = ranks
    return _rank_cache[anchor_id]


def is_novel(anchor_id, candidate_id):
    ranks = p1_rank_vec(anchor_id)
    cid = int(candidate_id)
    if ranks is None or cid not in p1_pid_to_idx:
        return None
    return int(ranks[p1_pid_to_idx[cid]]) > 10


# ---- Load this run's judgments, scoped to the anchors + configs judged here ----
results_df = pd.DataFrame([json.loads(l) for l in open(RESULTS_PATH)])
results_df = results_df[results_df["anchor_id"].isin(judged_anchors)].reset_index(drop=True)
clean = results_df[~results_df["parse_error"]].copy()

# ---- Macro-average per configuration (keyed on label, unique per config) ----
summary_rows = []
for c in CONFIGS:
    m = clean[clean["label"] == c["label"]]
    per_anchor_prec = []
    per_anchor_novel = []
    for aid, grp in m.groupby("anchor_id"):
        confirmed = grp[grp["is_substitute"] == True]
        per_anchor_prec.append(len(confirmed) / len(grp))               # Precision@10 for this anchor
        novs = [is_novel(aid, cid) for cid in confirmed["candidate_id"]]
        novs = [x for x in novs if x is not None]
        if novs:
            per_anchor_novel.append(sum(novs) / len(novs))
    summary_rows.append({
        "label": c["label"],
        "method": c["method"],
        "value": c["value"],
        "n_anchors_judged": int(m["anchor_id"].nunique()),
        "n_pairs_judged": int(len(m)),
        "precision_at_10": round(float(np.mean(per_anchor_prec)), 4) if per_anchor_prec else None,
        "novel_substitute_rate": round(float(np.mean(per_anchor_novel)), 4) if per_anchor_novel else None,
    })

summary_df = pd.DataFrame(summary_rows)[
    ["label", "n_anchors_judged", "n_pairs_judged", "precision_at_10", "novel_substitute_rate"]]
print(summary_df.to_string(index=False))

summary = {
    "macro_average": True,
    "n_eval_anchors": len(eval_anchors),
    "n_judged_anchors": int(len(judged_anchors)),
    "n_parse_errors": int(results_df["parse_error"].sum()),
    "configs": summary_rows,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSaved {SUMMARY_PATH.name}")

             label  n_anchors_judged  n_pairs_judged  precision_at_10  novel_substitute_rate
pct_dropoff (0.03)                11             110           0.0818                 0.6667
pct_dropoff (0.05)                11             110           0.1818                 0.7500
pct_dropoff (0.10)                11             110           0.1727                 0.6667
     kneedle (0.3)                11             110           0.2636                 0.7917
     kneedle (0.5)                11             110           0.2636                 0.7125
     kneedle (1.0)                11             110           0.2364                 0.7091

Saved summary.json


## C.5 Judge reliability spot-check

15 sampled judged pairs for a manual read-through (a smaller sample given the smaller total volume — the original drew 30 from 4,040). It uses the **same oversampling** of organic/certification pairs as the original, since that was the exact failure mode the prompt fix corrected; the cert target is scaled from the original's 10-of-30 to 5-of-15 to keep the same proportion. Each entry shows the anchor, candidate, configuration, the judge's reasoning, and its resolved verdict.

In [12]:
import re

CERT_PAT = re.compile(r"organic|natural|cage[- ]?free", re.IGNORECASE)


def _is_cert(row):
    return bool(CERT_PAT.search(str(row["anchor_name"])) or CERT_PAT.search(str(row["candidate_name"])))


spot_rng = np.random.default_rng(0)

sp = results_df.reset_index(drop=True)
cert_idx = [i for i in sp.index if _is_cert(sp.loc[i])]
cert_set = set(cert_idx)
other_idx = [i for i in sp.index if i not in cert_set]

TARGET = 15                              # scaled down from the original's 30
n_cert = min(5, len(cert_idx))           # ~1/3 organic/cert oversample, as in the original
pick = list(spot_rng.choice(cert_idx, size=n_cert, replace=False)) if n_cert else []
n_other = min(TARGET - len(pick), len(other_idx))
pick += list(spot_rng.choice(other_idx, size=n_other, replace=False)) if n_other else []
# Top up with any remaining cert pairs if still short of TARGET.
if len(pick) < TARGET:
    leftover = [i for i in cert_idx if i not in set(pick)]
    extra = min(TARGET - len(pick), len(leftover))
    if extra:
        pick += list(spot_rng.choice(leftover, size=extra, replace=False))

n_cert_in_pick = sum(1 for i in pick if i in cert_set)


def _verdict_txt(v):
    return "SUBSTITUTE" if v is True else ("NOT-a-substitute" if v is False else "PARSE_ERROR")


with open(SPOTCHECK_PATH, "w") as f:
    f.write(f"LLM judge reliability spot-check — {len(pick)} pairs "
            f"({n_cert_in_pick} organic/certification-oversampled)\n")
    f.write("=" * 92 + "\n\n")
    for j, i in enumerate(pick, 1):
        r = sp.loc[int(i)]
        f.write(f"[{j:>2}] config={r['label']}   (Step 1 novel={is_novel(r['anchor_id'], r['candidate_id'])})\n")
        f.write(f"     anchor    : {r['anchor_name']} ({r['anchor_id']})\n")
        f.write(f"     candidate : {r['candidate_name']} ({r['candidate_id']})\n")
        f.write(f"     reason    : {r['reason']}\n")
        f.write(f"     directional={r['is_directional_pair']}  restricted_item={r['restricted_item']}  "
                f"raw_is_substitute={r['raw_is_substitute']}\n")
        f.write(f"     VERDICT   : {_verdict_txt(r['is_substitute'])}\n\n")

print(f"Wrote {len(pick)} spot-check pairs to {SPOTCHECK_PATH.name} "
      f"({n_cert_in_pick} organic/certification pairs)")

Wrote 15 spot-check pairs to spotcheck.txt (5 organic/certification pairs)
